# Tutorial 5: Agentic Interpretability

This notebook teaches you how to use mlxterp's agentic tools — pre-built workflows, the AutoInterp ratchet loop, automated feature labeling, and report generation.

**What you'll learn:**
- Research workflows (`behavior_localization`, `circuit_discovery`, `feature_investigation`)
- Setting up an AutoInterp project (Karpathy AutoResearch-style)
- The experiment logging system
- Automated SAE feature labeling with Claude
- Generating shareable Markdown and HTML reports

**Prerequisites:** Tutorials 1-4. For auto-labeling: `pip install anthropic` + API key.

## 1. Setup

In [ ]:
from mlxterp import InterpretableModel
from mlxterp.workflows import behavior_localization, circuit_discovery
from mlxterp.autointerpret import (
    init_autointerpret, AutoInterpret,
    MetricRegistry, ExperimentLog, ExperimentEntry,
)
from mlxterp.reports import generate_report, save_report
import mlx.core as mx

model = InterpretableModel("mlx-community/Llama-3.2-1B-Instruct-4bit")
print(f"Model: {len(model.layers)} layers")

## 2. Research Workflows

Workflows chain together multiple analysis tools into a single call. Think of them as "run a full investigation in one line."

In [ ]:
# Behavior Localization: which components matter for factual recall?
clean = "The Eiffel Tower is in"
corrupted = "The Colosseum is in"

result = behavior_localization(
    model, clean, corrupted,
    metric="l2",
    steps=["dla", "patch_mlp", "patch_attn"],  # Skip head-level for speed
    verbose=True,
)

In [ ]:
# The result has a narrative summary
print("Narrative:")
print(result.narrative)
print()

# Access individual steps
dla = result.get_step("dla")
if dla:
    print(f"DLA target: '{dla.target_token_str}'")

mlp = result.get_step("patch_mlp")
if mlp:
    print(f"Top MLP layers: {mlp.top_components(k=3)}")

In [ ]:
# Full markdown report
print(result.to_markdown())

In [ ]:
# Circuit Discovery workflow
circuit_result = circuit_discovery(
    model, clean, corrupted,
    threshold=0.01,
    steps=["attribution", "acdc"],
    verbose=True,
)

acdc = circuit_result.get_step("acdc")
if acdc:
    print(f"\nCircuit: {acdc.summary()}")
    print(f"Important components: {acdc.nodes}")

## 3. AutoInterp: The Ratchet Loop

Inspired by Karpathy's AutoResearch. Set up a project directory, then let an agent (or yourself) run experiments in a loop.

### 3.1 Generate the Scaffold

In [ ]:
import os
import tempfile

# Create scaffold in a temp directory (use a real path for real projects)
project_dir = os.path.join(tempfile.gettempdir(), "mlxterp_autointerpret_demo")

path = init_autointerpret(
    output_dir=project_dir,
    model_name="mlx-community/Llama-3.2-1B-Instruct-4bit",
    research_question="How does Llama-3.2-1B recall the capital of France?",
    max_experiments=50,
)

print(f"Project created at: {path}")
print("\nFiles created:")
for f in sorted(os.listdir(path)):
    print(f"  {f}")

In [ ]:
# Look at the program.md (the research brief)
with open(os.path.join(path, "program.md")) as f:
    print(f.read())

In [ ]:
# Look at CLAUDE.md (agent instructions)
with open(os.path.join(path, "CLAUDE.md")) as f:
    print(f.read())

### 3.2 Programmatic Runner

Instead of using Claude Code (zero-orchestration mode), you can run experiments programmatically:

In [ ]:
from mlxterp.causal import activation_patching, direct_logit_attribution

runner = AutoInterpret(
    model=model,
    clean=clean,
    corrupted=corrupted,
    output_dir=os.path.join(project_dir, "programmatic_run"),
    max_experiments=10,
)

# Experiment 1: DLA scan
entry1 = runner.run_experiment(
    name="dla_scan",
    fn=lambda: direct_logit_attribution(model, clean),
    hypothesis="Identify which layers write the answer token",
)
print(f"Exp 1: {entry1.conclusion[:100]}")

# Experiment 2: MLP patching
entry2 = runner.run_experiment(
    name="mlp_patching",
    fn=lambda: activation_patching(model, clean, corrupted, component="mlp"),
    hypothesis="Find which MLP layers are causally important",
)
print(f"Exp 2: {entry2.conclusion[:100]}")

# Experiment 3: Attention patching
entry3 = runner.run_experiment(
    name="attn_patching",
    fn=lambda: activation_patching(model, clean, corrupted, component="attn"),
    hypothesis="Find which attention layers are causally important",
)
print(f"Exp 3: {entry3.conclusion[:100]}")

In [ ]:
# View experiment history
print(runner.summary())
print(f"\nTotal experiments: {runner.n_experiments}")
print(f"Done: {runner.is_done}")

In [ ]:
# Check findings directory
findings_dir = os.path.join(project_dir, "programmatic_run", "findings")
if os.path.exists(findings_dir):
    for f in sorted(os.listdir(findings_dir)):
        print(f"  {f}")

### 3.3 MetricRegistry

In [ ]:
# Define metrics for your experiments
metrics = MetricRegistry()
metrics.register(
    "logit_diff",
    lambda p, c, co, **kw: float(p[0, kw['correct']] - p[0, kw['incorrect']]),
    description="Logit difference between correct and incorrect tokens",
)
metrics.register(
    "top1_correct",
    lambda p, c, co, **kw: float(mx.argmax(p[0]) == kw['correct']),
    description="Whether top-1 prediction is the correct token",
)

# Discover available metrics (agent reads this)
print("Available metrics:")
for m in metrics.list():
    print(f"  {m['name']}: {m['description']}")

## 4. Automated Feature Labeling

Use Claude to automatically label SAE features. This requires `pip install anthropic` and an API key.

**Note:** The cells below will fail if you don't have the Anthropic SDK installed or an API key set. That's expected — the functions gracefully return placeholder labels.

In [ ]:
from mlxterp.auto_interp import auto_label_feature, FeatureLabel

# Without API access, this returns a placeholder label
# With API access, it sends max-activating examples to Claude and gets a label
label = auto_label_feature(
    model, sae=None,  # Would need a real SAE here
    feature_id=42,
    texts=[],  # Would need real texts
    layer=10,
)
print(f"Feature {label.feature_id}: {label.label}")
print(f"Description: {label.description}")

In [ ]:
# Example of what a labeled feature looks like
example_label = FeatureLabel(
    feature_id=42,
    label="capital_cities",
    description="Activates on mentions of capital cities and their countries",
    confidence=0.85,
    evidence=[
        {"text": "The capital of France is Paris", "activation_value": 0.95, "token_position": 7},
        {"text": "Berlin is the capital of Germany", "activation_value": 0.88, "token_position": 0},
    ],
    sensitivity_passed=True,
)

print(f"Label: {example_label.label}")
print(f"Description: {example_label.description}")
print(f"Confidence: {example_label.confidence:.0%}")
print(f"Evidence: {len(example_label.evidence)} examples")
print(f"Sensitivity test: {'PASS' if example_label.sensitivity_passed else 'FAIL'}")

## 5. Report Generation

Generate shareable reports from any analysis.

In [ ]:
# Run an analysis to get results
from mlxterp.causal import activation_patching

patching_result = activation_patching(
    model, clean, corrupted,
    component="mlp", metric="l2",
)

# Generate markdown report
report = generate_report(
    patching_result,
    title="Factual Recall: MLP Analysis",
    description="Investigating which MLP layers are involved in recalling 'Paris' for 'The Eiffel Tower is in'.",
)

print(report)

In [ ]:
# Combine multiple results into one report
attn_result = activation_patching(
    model, clean, corrupted,
    component="attn", metric="l2",
)

combined_report = generate_report(
    [patching_result, attn_result],
    title="Complete Patching Analysis",
    description="MLP and attention patching comparison.",
    include_json=True,
)

print(combined_report[:500] + "...")

In [ ]:
# Save as HTML (with embedded plots)
import tempfile, os

html_path = os.path.join(tempfile.gettempdir(), "mlxterp_report.html")
save_report(
    [patching_result, attn_result],
    html_path,
    title="Investigation Report",
    include_plots=True,
)
print(f"HTML report saved to: {html_path}")
print(f"Open in browser to view.")

In [ ]:
# Workflow results also generate reports
workflow_result = behavior_localization(
    model, clean, corrupted,
    steps=["patch_mlp", "patch_attn"],
    verbose=False,
)

print(workflow_result.to_markdown())

## 6. Using Claude Code as Your Interp Agent

The most powerful mode: just open Claude Code and tell it what to investigate.

### Option A: Zero Orchestration (Recommended)

```bash
# 1. Generate the scaffold
python -c "from mlxterp.autointerpret import init_autointerpret; init_autointerpret()"

# 2. Open in Claude Code
cd autointerpret
claude

# 3. Tell it:
# > Read program.md and start investigating.
```

Claude Code will import mlxterp directly, design experiments, run them, and accumulate findings.

### Option B: Ad-Hoc Investigation

```
# In Claude Code, just say:
# > Using mlxterp, investigate how Llama-3.2-1B handles subject-verb agreement.
# > Run DLA, then activation patching, then find the circuit.
```

Claude Code has full access to `import mlxterp` — no MCP server needed.

### Why No MCP Server?

Claude Code runs Python directly. An MCP server would add:
- Extra process management
- JSON serialization overhead for tensors
- Server lifecycle complexity
- No benefit (the agent already has `import mlxterp`)

**The library IS the interface.**

## Summary

| Tool | Purpose | Code |
|------|---------|------|
| `behavior_localization()` | Multi-step pipeline: DLA → patching → heads | `from mlxterp.workflows import ...` |
| `circuit_discovery()` | Pipeline: attribution → activation → ACDC | `from mlxterp.workflows import ...` |
| `init_autointerpret()` | Generate 3-file contract scaffold | `from mlxterp.autointerpret import ...` |
| `AutoInterpret` | Programmatic experiment runner | `runner.run_experiment(...)` |
| `ExperimentLog` | JSONL experiment logging | `log.append(entry)` |
| `auto_label_feature()` | LLM-generated feature labels | `from mlxterp.auto_interp import ...` |
| `sensitivity_test()` | Validate feature labels | `from mlxterp.auto_interp import ...` |
| `generate_report()` | Markdown/HTML reports | `from mlxterp.reports import ...` |
| `save_report()` | Save report to file | `from mlxterp.reports import ...` |

---

**You've completed the entire mlxterp tutorial series!** You now have tools for:
- Causal patching (Tutorial 1)
- Circuit discovery (Tutorial 2)
- Generation with interventions (Tutorial 3)
- Conversation analysis (Tutorial 4)
- Agentic interpretability (Tutorial 5)